# Study Notes: PyTorch Fundamentals (Prerequisite for LLMs)

**Course context:** While the overarching course focuses on Large Language Models (LLMs)—encompassing concepts like attention mechanisms, transformers, and fine-tuning—this specific lecture is dedicated entirely to building the necessary programmatic foundation. Before implementing complex LLM architectures, you must master the fundamental building blocks of deep learning using PyTorch.

---

## 1. Tensors: The Backbone of PyTorch

Tensors are the fundamental data structures in PyTorch, functioning similarly to multidimensional arrays or matrices, but with added capabilities for GPU acceleration and automatic differentiation.

### 1.1 Tensor Dimensions and Shapes

Understanding tensor dimensions is crucial for avoiding shape mismatch errors during matrix multiplications in neural networks.

* **0-D Tensor (Scalar):** A single number. Created via `torch.tensor(1)`.
* **1-D Tensor (Vector):** A list of numbers. e.g., `torch.tensor([1, 2, 3])`.
* **2-D Tensor (Matrix):** A list of vectors.
* **Identifying Dimensions:** The number of dimensions (`ndim` attribute or `.dim()` method) corresponds to the number of opening square brackets `[` at the beginning of the printed tensor.
* **Size/Shape:** Accessed via `.shape` or `.size()`. It describes the number of elements along each dimension. For example, a shape of `(2, 3)` means 2 elements in dimension 0 (rows) and 3 elements in dimension 1 (columns).

### 1.2 Tensor Creation

PyTorch provides multiple ways to instantiate tensors depending on your initialization needs:

* **From Lists:** `torch.tensor(data)`
* **Empty/Uninitialized:** `torch.empty(size)`
* **Random Values:** `torch.rand(size)` (Uniform distribution between 0 and 1).
* **Random Integers:** `torch.randint(low, high, size)`
* **Constant Values:** `torch.zeros(size)`, `torch.ones(size)`
* **Sequential Values:** `torch.arange(end)` (Similar to Python's `range()`)
* **Cloning shapes:** `torch.ones_like(tensor)`, `torch.rand_like(tensor)` (Creates a new tensor with the same shape and data type as the input tensor).

### 1.3 Data Types and Devices

* **Data Types:** The default floating-point data type in PyTorch is `float32`. You can check this using the `.dtype` attribute. To convert types, use `.type()`, e.g., `tensor.type(torch.double)` or `tensor.type(torch.float64)`.
* **Devices (CPU vs. GPU):** Tensors can reside on either the CPU or the GPU (CUDA). PyTorch's primary advantage over NumPy is its native GPU support.
* **Check availability:** `torch.cuda.is_available()`
* **Move tensor:** `tensor.to('cuda')` or `tensor.to('cpu')`
* **Direct creation:** `torch.rand(size, device='cuda')`



### 1.4 Interacting with NumPy *(Highly Testable)*

You can convert NumPy arrays to PyTorch tensors in two ways, but they behave differently regarding memory allocation:

1. `torch.from_numpy(array)`: **Shares memory** with the original NumPy array. Modifying the resulting tensor modifies the original array, and vice versa.
2. `torch.tensor(array)`: **Creates a copy** of the data. Modifying one does not affect the other.

### 1.5 Tensor Operations and Manipulation

* **Arithmetic:** Element-wise operations use standard operators (`+`, `-`, `*`, `/`) or PyTorch functions (`torch.add()`, `torch.sub()`, etc.).
* **Shape Manipulation:**
* `.view(shape)`: Changes the shape of the tensor. **Important:** It shares the underlying memory with the original tensor.
* `.reshape(shape)`: Also changes the shape, but relies on contiguous memory. It may return a view *or* a copy depending on memory layout, introducing ambiguity.
* *Trick:* Using `-1` in `.view()` (e.g., `.view(-1, 10)`) tells PyTorch to automatically infer the size of that dimension based on the total number of elements.


* **Concatenation:** `torch.cat((tensor_A, tensor_B), dim=X)` joins tensors along a specific dimension. Concatenating two `(3, 3)` tensors along `dim=1` results in a `(3, 6)` tensor.
* **Reduction Operations:** Functions like `.sum()`, `.mean()`, and `.argmax()` reduce a tensor's dimensions.
* If you apply `.sum(dim=0)` on a tensor of shape `(2, 3, 5)`, the operation collapses dimension 0, resulting in a shape of `(3, 5)`.



---

## 2. Autograd: Automatic Differentiation

`autograd` is PyTorch's automatic differentiation engine, which is the mathematical core that powers neural network training (backpropagation).

### 2.1 The Computational Graph

When you perform operations on tensors with `requires_grad=True`, PyTorch dynamically builds a **computational graph** to track every operation.

* **Leaf Tensors:** Tensors initialized directly by the user (e.g., inputs, model weights) with `requires_grad=True`. Their `.is_leaf` attribute is `True`.
* **Non-Leaf Tensors:** Tensors created as a result of mathematical operations on leaf tensors. Even if you don't explicitly set `requires_grad=True` on them, they inherit it because the gradient needs to flow through them to reach the leaf nodes.

### 2.2 Computing Gradients

Consider a function $y = 3\sigma(x) + 5$.

1. Initialize $x$: `x = torch.tensor([2.0], requires_grad=True)`
2. Compute $y$: `y = 3 * torch.sigmoid(x) + 5`
3. Trigger backpropagation: `y.backward()`
4. Access gradient: The derivative $\frac{dy}{dx}$ is evaluated and stored in `x.grad`.

* **`grad_fn`:** Non-leaf tensors have a `.grad_fn` attribute pointing to the function used to create them (e.g., `AddBackward`, `MulBackward`). This traces the operations backward.

### 2.3 Critical Exceptions and Rules *(Highly Testable)*

* **Gradient Accumulation:** Calling `.backward()` multiple times accumulates (adds) the gradients in the leaf tensor's `.grad` attribute. To prevent this during training loops, you must manually reset gradients to zero using `tensor.grad = 0` (or `optimizer.zero_grad()` in network training).
* **Intermediate Gradients:** By default, PyTorch only saves gradients for **leaf nodes**. If you try to check `.grad` on an intermediate (non-leaf) node, you will get a warning, and it will return `None`. If you must inspect an intermediate gradient, call `.retain_grad()` on that non-leaf tensor *before* calling `.backward()`.
* **Disabling Autograd:** Use the context manager `with torch.no_grad():` to temporarily disable graph tracking. This is essential during model evaluation or when manually updating weights, as you don't want the weight-update math to become part of the computational graph itself.

---

## 3. Building a Neural Network Pipeline

Building an LLM or any basic Deep Learning model in PyTorch follows a structured, repeatable pipeline utilizing standard modules.

### 3.1 Key Modules

* `torch.nn`: Contains tools for building neural networks (Layers like `nn.Linear`, loss functions like `nn.CrossEntropyLoss`).
* `torch.optim`: Contains optimization algorithms (e.g., `optim.SGD`, `optim.Adam`).
* `torchvision`: A utility library for computer vision datasets (like MNIST) and image transformations.

### 3.2 Data Loading

Data is loaded in batches because entire datasets (e.g., 60,000 MNIST training images) cannot fit into memory simultaneously.

* **Dataset:** Holds the data and labels.
* **DataLoader:** Wraps an iterable around the Dataset. It handles **batching** (serving data in chunks, e.g., 32 images at a time) and **shuffling** (randomizing data order for training to prevent the model from learning the sequence).

### 3.3 Defining the Network Architecture

Custom models inherit from `nn.Module`.

1. `__init__(self, ...)`: Define the layers (e.g., `self.layer1 = nn.Linear(input_size, hidden_size)`). Parameters (weights and biases) inside `nn.Linear` automatically have `requires_grad=True`.
2. `forward(self, x)`: Defines how data flows through the layers (the forward pass). E.g., applying a ReLU activation between linear layers.

*Note on MNIST dimensions:* MNIST images are 2D (28x28 pixels). Before passing them to a standard linear layer, they must be flattened into a 1D tensor of size 784 ($28 \times 28$).

### 3.4 Loss and Optimization

* **Cross-Entropy Loss:** Used for multi-class classification. *Important:* PyTorch's `nn.CrossEntropyLoss` expects **raw, unnormalized logits**. You do *not* need to apply a Softmax layer manually in your `forward()` function; the loss function handles it internally for numerical stability.
* **Optimizer:** Updates the model's weights based on computed gradients. `optimizer = optim.Adam(model.parameters(), lr=0.001)`

### 3.5 The Standard Training Loop

You must memorize this exact sequence for training iterations:

1. **Forward Pass:** Pass input through the model: `predictions = model(inputs)`
2. **Compute Loss:** Compare predictions to ground truth: `loss = loss_fn(predictions, labels)`
3. **Zero Gradients:** Clear old accumulated gradients: `optimizer.zero_grad()` (or `w.grad=0`)
4. **Backward Pass:** Compute new gradients: `loss.backward()`
5. **Update Weights:** Adjust parameters: `optimizer.step()`

---

## NPTEL Practice Questions

**Q1. A PyTorch tensor `A` is created from a NumPy array `arr` using the code `A = torch.from_numpy(arr)`. If you multiply all elements in `arr` by 3, what happens to tensor `A`?**
a) Tensor `A` remains unchanged because it is a separate copy.
b) Tensor `A` elements are also multiplied by 3 because it shares the same memory as `arr`.
c) PyTorch throws a memory conflict error.
d) Tensor `A` is automatically deleted.
**Answer: b)** Explanation: `torch.from_numpy()` creates a tensor that shares the underlying memory space with the original NumPy array. Any modifications to one reflect in the other. If `torch.tensor(arr)` were used instead, it would have created an independent copy.

**Q2. Suppose you have a 3D tensor `X` with shape `(4, 5, 6)`. You perform the operation `Y = X.sum(dim=1)`. What is the resulting shape of tensor `Y`?**
a) (4, 6)
b) (5, 6)
c) (4, 5)
d) (1, 5, 1)
**Answer: a)** Explanation: In a reduction operation like `.sum()`, the specified dimension collapses. The original dimensions are index 0 (size 4), index 1 (size 5), and index 2 (size 6). Collapsing dimension 1 removes the "5", leaving `(4, 6)`.

**Q3. You define a variable `X` as a leaf tensor with `requires_grad=True`. You perform `Y = X * 2` and `Z = Y + 3`. You call `Z.backward()`. If you attempt to print `Y.grad`, what will happen?**
a) It will print the correct gradient of Z with respect to Y.
b) It will print 0.
c) It will return `None` (and likely throw a warning) because PyTorch defaults to clearing intermediate gradients to save memory.
d) It will crash the program entirely.
**Answer: c)** Explanation: `Y` is a non-leaf tensor. By default, PyTorch only retains gradients for leaf tensors (`X`). To access `Y.grad`, you must call `Y.retain_grad()` *before* calling `Z.backward()`.

**Q4. During the implementation of a neural network for multi-class classification in PyTorch, why do we pass raw output values directly to the Cross-Entropy loss function without applying a Softmax activation first?**
a) Softmax is computationally too expensive for PyTorch.
b) Softmax is irrelevant for multi-class classification.
c) PyTorch's `nn.CrossEntropyLoss` internally applies Softmax, doing so in a numerically stable way.
d) Raw values yield a lower loss by default.
**Answer: c)** Explanation: As explicitly stated in the lecture, the `CrossEntropyLoss` function in PyTorch takes raw, unnormalized outputs (logits) directly. It combines `LogSoftmax` and `NLLLoss` into one stable mathematical operation internally.

**Q5. In what specific step sequence should the operations in a standard PyTorch training loop be executed?**
a) Forward pass $\rightarrow$ Backward pass $\rightarrow$ Optimizer step $\rightarrow$ Compute Loss $\rightarrow$ Zero gradients
b) Forward pass $\rightarrow$ Compute Loss $\rightarrow$ Backward pass $\rightarrow$ Optimizer step $\rightarrow$ Zero gradients
c) Zero gradients $\rightarrow$ Forward pass $\rightarrow$ Optimizer step $\rightarrow$ Backward pass $\rightarrow$ Compute Loss
d) Optimizer step $\rightarrow$ Zero gradients $\rightarrow$ Forward pass $\rightarrow$ Compute Loss $\rightarrow$ Backward pass
**Answer: b)** Explanation: You must first pass data through the model (Forward), calculate the error (Compute Loss), calculate the gradients of the error with respect to weights (Backward), use those gradients to update the weights (Optimizer step), and ensure gradients don't carry over to the next loop (Zero gradients). Note: Zeroing gradients can safely happen right before the backward pass as well, but the sequence in 'b' represents standard logical flow.

---

## Revision Summary

* **Dimensions:** Found by counting outer brackets (`[`). Accessed via `.ndim` or `.dim()`.
* **Memory:** `.from_numpy()` and `.view()` **share** memory. `torch.tensor()` creates a **copy**.
* **Device Management:** Use `.to(device)` or specify `device='cuda'` on creation.
* **Reductions:** `dim=X` identifies the dimension that will disappear after operations like `.sum()`.
* **Leaf Tensors:** Tensors you initialize with `requires_grad=True`.
* **Accumulation:** PyTorch adds gradients iteratively. Always zero them via `optimizer.zero_grad()` or `x.grad=0` in training loops.
* **Intermediate Gradients:** Use `.retain_grad()` to capture gradients of non-leaf nodes.
* **Context Managers:** Use `with torch.no_grad():` for manual weight updates or inference to prevent tracking in the computational graph.
* **`nn.Module`:** Base class for all custom models. Implement layers in `__init__` and pass logic in `forward`.
* **Data loaders:** Used to serve large data in shuffled, manageable batches.